# Experiment 2: history length

**Question:** how does the forecast change if TiRex-2 gets 1, 2, ..., 10 weeks of temperature history instead
of the 4 weeks we use so far?

**Setup**
- Data: Austria 15 km (team dataset), temperature only, no time features. Forecast: 24 h.
- History: 1 to 10 weeks (168 to 1,680 h; the model accepts up to 2,048 h).
- Cases: 50 random points × 48 forecast starts (about one per week) spread over the year (mid-December to end of September, so that
  10 weeks of history exist), start hours 00 / 06 / 12 / 18 UTC in turn. Every case is forecast with all 10 lengths.

**Measures** (both in °C, averaged over the 24 forecast hours)
- **error**: absolute difference between the forecast (median) and what really happened
- **band width**: width of the 10-90 % band, i.e. how unsure the model is

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots
from tirex2 import TimeseriesType, load_model

DATA = Path("/workspace/data/austria_15km")
FIG = Path("/workspace/phase1-model-behavior/pair-1/experiments/figures/exp2")
FIG.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("/workspace/data/notebooks/results")
RERUN = False  # True = run the model again even if saved forecasts exist
H = 24
WEEKS = list(range(1, 11))
N_POINTS, N_STARTS = 50, 48
if os.environ.get("QUICK"):  # quick code test
    N_POINTS, N_STARTS = 3, 2
    FIG = Path("/tmp")  # test runs must not overwrite the real figures
RESULTS = RESULTS_DIR / f"exp2_forecasts_{N_POINTS}x{N_STARTS}.npz"
COLORS = sample_colorscale("Viridis", [i / (len(WEEKS) - 1) for i in range(len(WEEKS))])  # 1 week = dark ... 10 weeks = yellow

In [2]:
df = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", "T2M"])
T = df.pivot(index="time", columns="point_id", values="T2M").interpolate(limit=3, limit_area="inside")
times, Y = T.index, T.to_numpy(np.float32)
col = {p: i for i, p in enumerate(T.columns)}

rng = np.random.default_rng(0)
pids = np.sort(rng.choice(T.columns, N_POINTS, replace=False))
days = pd.date_range(times[0] + pd.Timedelta(weeks=10, days=1), times[-1] - pd.Timedelta(days=2), periods=N_STARTS).normalize()
starts = [d + pd.Timedelta(hours=6 * (i % 4)) for i, d in enumerate(days)]
cases = pd.DataFrame([(p, s, times.get_loc(s)) for s in starts for p in pids], columns=["point_id", "start", "o"])
truth = np.stack([Y[o : o + H, col[p]] for p, o in zip(cases.point_id, cases.o)])
print(f"{len(pids)} points x {len(starts)} starts = {len(cases)} cases, {len(cases) * len(WEEKS)} forecasts")
print("starts:", ", ".join(f"{s:%Y-%m-%d %H}h" for s in starts))

50 points x 48 starts = 2400 cases, 24000 forecasts
starts: 2025-12-11 00h, 2025-12-17 06h, 2025-12-23 12h, 2025-12-29 18h, 2026-01-04 00h, 2026-01-11 06h, 2026-01-17 12h, 2026-01-23 18h, 2026-01-29 00h, 2026-02-04 06h, 2026-02-11 12h, 2026-02-17 18h, 2026-02-23 00h, 2026-03-01 06h, 2026-03-07 12h, 2026-03-14 18h, 2026-03-20 00h, 2026-03-26 06h, 2026-04-01 12h, 2026-04-08 18h, 2026-04-14 00h, 2026-04-20 06h, 2026-04-26 12h, 2026-05-02 18h, 2026-05-09 00h, 2026-05-15 06h, 2026-05-21 12h, 2026-05-27 18h, 2026-06-02 00h, 2026-06-09 06h, 2026-06-15 12h, 2026-06-21 18h, 2026-06-27 00h, 2026-07-03 06h, 2026-07-10 12h, 2026-07-16 18h, 2026-07-22 00h, 2026-07-28 06h, 2026-08-04 12h, 2026-08-10 18h, 2026-08-16 00h, 2026-08-22 06h, 2026-08-28 12h, 2026-09-04 18h, 2026-09-10 00h, 2026-09-16 06h, 2026-09-22 12h, 2026-09-28 18h


In [3]:
model = load_model("NX-AI/TiRex-2", device="cpu")


def forecast(weeks):
    """(cases, 9 quantiles, H) forecasts with `weeks` weeks of history."""
    L = weeks * 168
    ts = [TimeseriesType(torch.from_numpy(Y[o - L : o, col[p]][None].copy()), None, None) for p, o in zip(cases.point_id, cases.o)]
    out = model.forecast(ts, prediction_length=H, output_type="numpy", batch_size=64)
    return np.stack([q[0] for q in out])


fc, secs = {}, {}
if RESULTS.exists() and not RERUN and not os.environ.get("QUICK"):
    z = np.load(RESULTS)
    assert list(z["point_id"]) == list(cases.point_id)
    fc = {w: z[f"w{w}"] for w in WEEKS}
    secs = dict(zip(WEEKS, z["seconds"]))
    print(f"loaded saved forecasts from {RESULTS.name}")
else:
    for w in WEEKS:
        t0 = time.time()
        fc[w] = forecast(w)
        secs[w] = time.time() - t0
        print(f"{w:2d} weeks: {secs[w]:5.0f} s ({1000 * secs[w] / len(cases):.0f} ms per forecast)")
    if not os.environ.get("QUICK"):
        RESULTS.parent.mkdir(exist_ok=True)
        np.savez_compressed(RESULTS, point_id=cases.point_id.to_numpy(), start=cases.start.astype(str).to_numpy().astype("U"), truth=truth,
                            seconds=np.array([secs[w] for w in WEEKS]),
                            **{f"w{w}": fc[w] for w in WEEKS})

full = 378 * len(starts) * 1000 * sum(secs.values()) / len(cases) / 1000 / 60
print(f"estimated time for all 378 points with the same {len(starts)} starts: {full:.0f} min")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

loaded saved forecasts from exp2_forecasts_50x48.npz
estimated time for all 378 points with the same 48 starts: 231 min


## Error and band width per history length

In [4]:
err = {w: np.abs(fc[w][:, 4] - truth) for w in WEEKS}  # (cases, H)
width = {w: fc[w][:, 8] - fc[w][:, 0] for w in WEEKS}
summary = pd.DataFrame({
    "error (°C)": [err[w].mean() for w in WEEKS],
    "band width (°C)": [width[w].mean() for w in WEEKS],
    "seconds": [secs[w] for w in WEEKS],
}, index=pd.Index(WEEKS, name="weeks of history"))
summary.round(2)

,error (°C),band width (°C),seconds
weeks of history,,,
1,1.85,5.41,174.81
2,1.79,5.31,172.49
3,1.78,5.36,177.13
4,1.77,5.33,183.76
5,1.76,5.38,172.64
6,1.76,5.40,172.92
7,1.78,5.38,202.32
8,1.79,5.36,191.22
9,1.79,5.39,187.10


### All cases

Line: mean over all cases. Band: middle half of the cases (25-75 %). Dotted line: 4 weeks, the setup used so far.

In [5]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=["Error: how far off is the forecast?", "Band width: how unsure is the model?"])
for c, (vals, name) in enumerate([(err, "error"), (width, "band width")], start=1):
    per_case = np.stack([vals[w].mean(axis=1) for w in WEEKS])  # (weeks, cases)
    q25, q75 = np.percentile(per_case, [25, 75], axis=1)
    fig.add_trace(go.Scatter(x=np.r_[WEEKS, WEEKS[::-1]], y=np.r_[q75, q25[::-1]], mode="lines", fill="toself",
                             fillcolor="rgba(31,119,180,0.15)", line={"width": 0}, hoverinfo="skip", showlegend=False), row=1, col=c)
    fig.add_trace(go.Scatter(x=WEEKS, y=per_case.mean(axis=1), mode="lines+markers", line={"color": "#1f77b4", "width": 3},
                             marker={"color": "#1f77b4", "size": 7}, showlegend=False,
                             hovertemplate="%{x} weeks: mean " + name + " %{y:.2f} °C<extra></extra>"), row=1, col=c)
    fig.add_vline(x=4, line={"color": "grey", "dash": "dot"}, row=1, col=c)
    fig.update_xaxes(title_text="weeks of history", dtick=1, row=1, col=c)
    fig.update_yaxes(title_text=f"{name} (°C)", rangemode="tozero", row=1, col=c)
fig.update_layout(height=420, template="plotly_dark",
                  title=f"Does a longer history help?<br><sup>{len(cases)} cases ({len(pids)} points × {len(starts)} starts), "
                        "24 h forecast, averaged over the 24 hours</sup>")
fig.write_html(FIG / "exp2_error_vs_history.html", include_plotlyjs="cdn")
fig.show()

### One typical case

The case with the median error at 4 weeks (typical, not hand-picked): the last 3 days of history, what happened
(white), and the 10 forecasts (dark = 1 week of history ... yellow = 10 weeks). Click legend entries to hide lines.

In [6]:
k = int(np.argsort(err[4].mean(axis=1))[len(cases) // 2])
p, s, o = cases.loc[k, ["point_id", "start", "o"]]
hist_t, fut_t = times[o - 72 : o], times[o : o + H]
fig = go.Figure()
fig.add_trace(go.Scatter(x=hist_t, y=Y[o - 72 : o, col[p]], name="history", line={"color": "grey", "width": 1.5}))
fig.add_trace(go.Scatter(x=fut_t, y=truth[k], name="what happened", line={"color": "white", "width": 2.5}))
for w, c in zip(WEEKS, COLORS):
    fig.add_trace(go.Scatter(x=fut_t, y=fc[w][k, 4], name=f"{w} week{'s' if w > 1 else ''}", line={"color": c, "width": 2},
                             hovertemplate="%{x|%d %b %H:%M}: %{y:.1f} °C<extra>" + f"{w} weeks" + "</extra>"))
fig.add_vline(x=fut_t[0], line={"color": "grey", "dash": "dot"})
fig.update_layout(height=450, template="plotly_dark",
                  title=f"Point {p}, forecast from {s:%Y-%m-%d %H:%M} UTC with 1-10 weeks of history",
                  xaxis={"title": None, "tickformat": "%a %d %b\n%H:%M"}, yaxis={"title": "T2M (°C)"})
fig.write_html(FIG / "exp2_example_case.html", include_plotlyjs="cdn")
fig.show()

## Does an input that encodes the time help?

Same cases, 4 weeks of history. Besides the temperature, the model gets the time as **future covariates** (known
for the history and the 24 forecast hours), encoded as a circle (sine and cosine) so that 23:00 and 00:00 are close:

| Setup | Extra input |
|---|---|
| **temperature only** | nothing (the 4-week forecasts from above) |
| **+ hour of day** | hour of day (2 series) |
| **+ hour of day + time of year** | hour of day and day of the year (4 series) |

In [7]:
L = 4 * 168
TIME_SETUPS = {"none": "temperature only", "hour": "+ hour of day", "hour_year": "+ hour of day + time of year"}
TIME_COLORS = {"none": "#7f7f7f", "hour": "#1f77b4", "hour_year": "#ff7f0e"}
RES_TIME = RESULTS.with_name(RESULTS.name.replace("forecasts", "time_inputs"))


def time_features(t, year):
    h = 2 * np.pi * np.asarray(t.hour) / 24
    f = [np.sin(h), np.cos(h)]
    if year:
        d = 2 * np.pi * np.asarray(t.day_of_year) / 365.25
        f += [np.sin(d), np.cos(d)]
    return np.stack(f).astype(np.float32)


ft = {"none": fc[4]}
if RES_TIME.exists() and not RERUN and not os.environ.get("QUICK"):
    zt = np.load(RES_TIME)
    assert list(zt["point_id"]) == list(cases.point_id)
    ft.update({s: zt[s] for s in ["hour", "hour_year"]})
    print(f"loaded saved forecasts from {RES_TIME.name}")
else:
    for s in ["hour", "hour_year"]:
        t0 = time.time()
        ts = [TimeseriesType(torch.from_numpy(Y[o - L : o, col[p]][None].copy()), None,
                             torch.from_numpy(time_features(times[o - L : o + H], s == "hour_year")))
              for p, o in zip(cases.point_id, cases.o)]
        out = model.forecast(ts, prediction_length=H, output_type="numpy", batch_size=64)
        ft[s] = np.stack([q[0] for q in out])
        print(f"{TIME_SETUPS[s]:32s} {time.time() - t0:5.0f} s ({1000 * (time.time() - t0) / len(cases):.0f} ms per forecast)")
    if not os.environ.get("QUICK"):
        np.savez_compressed(RES_TIME, point_id=cases.point_id.to_numpy(), start=cases.start.astype(str).to_numpy().astype("U"),
                            truth=truth, **{s: ft[s] for s in ["hour", "hour_year"]})

terr = {s: np.abs(ft[s][:, 4] - truth) for s in TIME_SETUPS}
pd.DataFrame({"error (°C)": [terr[s].mean() for s in TIME_SETUPS],
              "band width (°C)": [(ft[s][:, 8] - ft[s][:, 0]).mean() for s in TIME_SETUPS],
              "cases better than temperature only": [np.nan] + [f"{(terr[s].mean(1) < terr['none'].mean(1)).mean():.0%}"
                                                            for s in ["hour", "hour_year"]]},
             index=pd.Index(TIME_SETUPS.values(), name="setup")).round(2)

loaded saved forecasts from exp2_time_inputs_50x48.npz


,error (°C),band width (°C),cases better than temperature only
setup,,,
temperature only,1.77,5.33,NaN
+ hour of day,1.78,5.47,53%
+ hour of day + time of year,1.77,5.44,52%


In [8]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.12,
                    subplot_titles=["Error by hours ahead (mean over all cases)",
                                    "Change in error per case (time input minus temperature only)"])
for s, label in TIME_SETUPS.items():
    fig.add_trace(go.Scatter(x=np.arange(1, H + 1), y=terr[s].mean(axis=0), mode="lines", name=label,
                             line={"color": TIME_COLORS[s], "width": 2.5},
                             hovertemplate="%{x} h ahead: %{y:.2f} °C<extra>" + label + "</extra>"), row=1, col=1)
for s in ["hour", "hour_year"]:
    d = terr[s].mean(axis=1) - terr["none"].mean(axis=1)
    fig.add_trace(go.Box(x=d, name=TIME_SETUPS[s], marker={"color": TIME_COLORS[s], "size": 3}, line={"color": TIME_COLORS[s]},
                         boxpoints="all", jitter=0.4, pointpos=0, showlegend=False, orientation="h",
                         hovertemplate="%{x:+.2f} °C<extra></extra>"), row=1, col=2)
fig.add_vline(x=0, line={"color": "white", "width": 1}, row=1, col=2)
fig.update_xaxes(title_text="hours ahead", dtick=2, row=1, col=1)
fig.update_yaxes(title_text="error (°C)", rangemode="tozero", row=1, col=1)
fig.update_xaxes(title_text="°C (left of 0 = time input helps)", row=1, col=2)
fig.update_layout(height=460, template="plotly_dark", legend={"orientation": "h", "y": -0.2},
                  title=f"Does telling the model the time help?<br><sup>{len(cases)} cases, 4 weeks of history, "
                        "24 h forecast. Right: one dot per case.</sup>")
fig.write_html(FIG / "exp2_time_inputs.html", include_plotlyjs="cdn")
fig.show()

## Reusing the results

All forecasts are saved in `data/notebooks/results/exp2_forecasts_50x48.npz`, so other visualizations can use
them without running the model again:

```python
z = np.load("/workspace/data/notebooks/results/exp2_forecasts_50x48.npz")
z["point_id"], z["start"]   # one entry per case (point, forecast start in UTC)
z["truth"]                  # (cases, 24) what really happened, °C
z["w4"]                     # (cases, 9 quantiles 10-90 %, 24 h) forecasts with 4 weeks of history; w1 ... w10
```

The time-input forecasts (same cases, 4 weeks) are in `exp2_time_inputs_50x48.npz`: `z["hour"]`, `z["hour_year"]`.